In [ ]:
%%capture
import os
from django_pandas.io import read_frame
from edc_model_to_dataframe import read_frame_edc
from pathlib import Path
import pandas as pd

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
documents_folder = os.environ["INTECOMM_DOCUMENTS_FOLDER"]
plus = activate(dotenv_file=env_file)

report_folder = Path(documents_folder)


Assumptions
1. endline BP is >=9m post-baseline (check inte-africa)
2. 140 or 90
3. 180 or 110
4. use one measure if two were not taken
5. all initial BP measurements are taken at baseline or within 30 days of baseline
6. duration between baseline and endline measurements must be at least 9m


In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858
from datetime import timedelta


df_main = get_df_main_1858(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/"))
# df_main = pd.read_csv(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/") / "df_main_1858.csv")

In [ ]:
df_main["onstudy_days"].describe()

In [ ]:
df_main[(df_main.hiv_only==1)][["onstudy_days"]]

In [ ]:
df_main[(df_main.ncd==1)][["onstudy_days"]]

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import shapiro, probplot
import seaborn as sns
import scipy.stats as stats

df_a = df_main[(df_main.hiv_only==1)][["onstudy_days"]]
df_b = df_main[(df_main.ncd==1)][["onstudy_days"]]

# Plot histograms and Q-Q plots to visually inspect the distribution
plt.figure(figsize=(12, 6))

plt.subplot(2, 2, 1)
sns.histplot(df_a, kde=True)
plt.title('Histogram of Interval Days (Group A)')

plt.subplot(2, 2, 2)
sns.histplot(df_b, kde=True)
plt.title('Histogram of Interval Days (Group B)')

# Plot Q-Q plots to visually inspect the distribution
plt.subplot(2, 2, 3)
probplot(df_a, dist="norm", plot=plt)
plt.title('Q-Q Plot of Interval Days (Group A)')

plt.subplot(2, 2, 4)
probplot(df_b, dist="norm", plot=plt)
plt.title('Q-Q Plot of Interval Days (Group B)')

plt.tight_layout()
plt.show()

In [ ]:
# assert all BP measurements are taken at baseline or within 30 days of baseline
df1 = df_main[["subject_identifier", "htn", "baseline_datetime", "bp_datetime_baseline"]].copy()
df1.reset_index(inplace=True, drop=True)
df1.loc[:, "interval"] = df1["bp_datetime_baseline"] - df1["baseline_datetime"]
df1[(df1.htn==1) & (df1["interval"]>timedelta(days=0))]["subject_identifier"].count() == 0


In [ ]:
# inspect count of subjects with measurements less than 9m apart
days = 30 * 9
df1 = df_main[(df_main.htn==1) & (df_main.bp_measured_delta<timedelta(days=days))][["subject_identifier", "site", "bp_measured_delta", "bp_visit_code_baseline", "bp_visit_code_endline", "endline_visit_code", "offstudy_reason"]].copy()
df1.reset_index(drop=True, inplace=True)
df1.offstudy_reason.value_counts()


In [ ]:
from edc_pdutils.dataframes import get_subject_visit

df_visit = get_subject_visit("intecomm_subject.subjectvisit")
# df_visit[df_visit.].groupby("subject_identifier").size()
df_visit.appt_status.value_counts()

In [ ]:
from edc_appointment.models import Appointment
from intecomm_subject.models import SubjectVisit

df= read_frame(Appointment.objects.all())
df[df.visit_code_sequence==0].appt_status.value_counts()

In [ ]:

# cond = (df_main.htn==1)
cond = (df_main.subject_identifier.notna())

In [ ]:
df_a_baseline = df_main[cond & (df_main.assignment=="a")].agg({
    'bp_sys_baseline': ['count','mean', 'median', 'min', 'max', 'std'],
    'bp_dia_baseline': ['count','mean', 'median', 'min', 'max', 'std']
}).transpose().reset_index()
df_a_baseline["assignment"] = "a"


In [ ]:
df_a_endline = df_main[cond &  (df_main.assignment=="a")].agg({
    'bp_sys_endline': ['count', 'mean', 'median', 'min', 'max', 'std'],
    'bp_dia_endline': ['count', 'mean', 'median', 'min', 'max', 'std']
}).transpose().reset_index()
df_a_endline["assignment"] = "a"


In [ ]:
df_b_baseline = df_main[cond &  (df_main.assignment=="b")].agg({
    'bp_sys_baseline': ['count','mean', 'median', 'min', 'max', 'std'],
    'bp_dia_baseline': ['count','mean', 'median', 'min', 'max', 'std']
}).transpose().reset_index()
df_b_baseline["assignment"] = "b"


In [ ]:
df_b_endline = df_main[cond &  (df_main.assignment=="b")].agg({
    'bp_sys_endline': ['count', 'mean', 'median', 'min', 'max', 'std'],
    'bp_dia_endline': ['count', 'mean', 'median', 'min', 'max', 'std']
}).transpose().reset_index()
df_b_endline["assignment"] = "b"


In [ ]:
df_bp = pd.concat([df_a_baseline, df_b_baseline, df_a_endline,  df_b_endline])
df_bp

In [ ]:
# >=140/90


df_main[(df_main.htn==1) & (df_main.assignment=="a") & ((df_main.bp_sys_baseline>=140) | (df_main.bp_dia_baseline>=90))]

In [ ]:
df_main[(df_main.htn==1) & (df_main.assignment=="a") & ((df_main.bp_sys_endline>=140) | (df_main.bp_dia_endline>=90))]


In [ ]:
# baseline controlled
df_main[(df_main.htn==1) & (df_main.bp_controlled_baseline==1)].groupby(by=["assignment"]).size()



In [ ]:
# endline controlled
df_main[(df_main.htn==1) & (df_main.bp_controlled_endline==1)].groupby(by=["assignment"]).size()

In [ ]:
# baseline severe htn
df_main[(df_main.htn==1) & (df_main.bp_severe_htn_baseline==1)].groupby(by=["assignment"]).size()


In [ ]:
# endline severe htn
df_main[(df_main.htn==1) & (df_main.bp_severe_htn_endline==1)].groupby(by=["assignment"]).size()
